# P08 — La atención es todo lo que necesitas

## 1. Título y paper

**Paper:** *Attention Is All You Need*  
**Autoría:** Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin  
**Año y venue:** 2017 · arXiv:1706.03762 · NeurIPS (NIPS) 2017  
**Nivel:** L4 · **Motor:** `transformer`  
**Ficha completa:** [`P08_transformer`](../../papers/foundational/P08_transformer/README.md)

**Hito:** Elimina la recurrencia y la convolución del modelado de secuencias: todo el cómputo de una capa se paraleliza.

- [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
- [NeurIPS 2017 (proceedings)](https://papers.nips.cc/paper_files/paper/2017)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: La recurrencia impone un cómputo secuencial en la longitud de la secuencia y camina O(n) pasos entre posiciones distantes; eso limita el entrenamiento a gran escala.
2. Ejecutar una implementación mínima de la propuesta: Un encoder–decoder compuesto solo de self-attention multi-cabeza, redes feed-forward por posición, conexiones residuales, layer normalization y codificación posicional.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P07
- P04


## 4. Intuición

Cada palabra pregunta al resto de la frase «¿quién de vosotros me importa?», recibe una respuesta ponderada y se actualiza. Todas las palabras lo hacen **a la vez**, no en fila. Ahí está la paralelización, y ahí está el salto de escala.


## 5. Concepto mínimo

```text
Attention(Q, K, V) = softmax(QKᵀ / √d_k) · V
MultiHead(X) = Concat(head₁ … head_h)·W^O,  head_i = Attention(XW_i^Q, XW_i^K, XW_i^V)
PE(pos, 2i) = sin(pos/10000^{2i/d}),  PE(pos, 2i+1) = cos(pos/10000^{2i/d})
sublayer(x) = LayerNorm(x + Sublayer(x))
```

`√d_k` no es cosmética: sin ella el producto escalar crece con la dimensión, el softmax se satura y el gradiente se apaga.


## 6. Código explicado

El motor implementa la ecuación 1 completa: escala, máscara causal, multi-cabeza, codificación posicional y residual + layer norm.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('transformer', seed=7)['result']
show(r['entropia_media'])
print('\nmatriz de atención con máscara causal:')
for fila in r['mascara_causal']:
    print(' ', fila)

## 7. Predicción antes de ejecutar

1. ¿La entropía será mayor con escala `√d_k` o sin ella? ¿Qué significa cada caso?
2. ¿Qué forma tendrá la matriz con máscara causal?
3. Para n=1000, ¿cuántas veces más operaciones hace la self-attention que la recurrencia con d=8?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for fila in r['complejidad']:
    print(f"n={fila['n']:>5} · self-attention {fila['self_attention_ops']:>10} ops "
          f"· recurrente {fila['recurrent_ops']:>8} ops "
          f"· camino RNN {fila['camino_maximo_rnn']:>5} vs attention {fila['camino_maximo_attention']}")

## 9. Salida interpretable

Dos lecturas opuestas y ambas ciertas: el camino entre dos posiciones es **1** en atención y **n** en recurrencia (ventaja de optimización), pero el coste crece con **n²** (desventaja de memoria y cómputo). El Transformer compró paralelismo pagando con complejidad cuadrática.


## 10. Comentario pedagógico

El título es una consigna, no un teorema. El modelo del paper **también** necesita redes feed-forward por posición, residuales, layer norm y codificación posicional. Sin ellas la atención sola no entrena. Esta es la sección 3 del paper, no una interpretación.


## 11. Error o anti-patrón deliberado

Anti-patrón deliberado: quitar la escala `√d_k` y creer que «da casi igual».


In [ ]:
import math
import random

rng = random.Random(0)
for d_k in (4, 64, 512):
    q = [rng.gauss(0, 1) for _ in range(d_k)]
    k = [rng.gauss(0, 1) for _ in range(d_k)]
    crudo = sum(a * b for a, b in zip(q, k))
    print(f'd_k={d_k:>3} · qᵀk sin escalar = {crudo:+8.2f} · escalado = {crudo / math.sqrt(d_k):+6.2f}')

## 12. Corrección

La magnitud del producto escalar crece como √d_k. Al dividir por √d_k, la varianza vuelve a ~1 y el softmax no se satura:


In [ ]:
def softmax(xs):
    m = max(xs)
    e = [math.exp(x - m) for x in xs]
    return [v / sum(e) for v in e]

scores = [12.0, 10.5, 9.0, 8.0]
print('sin escalar :', [round(p, 4) for p in softmax(scores)])
print('escalado /8 :', [round(p, 4) for p in softmax([s / 8 for s in scores])])
print('→ sin escalar, un solo token acapara casi toda la masa y el gradiente del resto ≈ 0')

## 13. Desafío guiado

Verifica que la codificación posicional distingue posiciones y que posiciones cercanas tienen codificaciones parecidas.


In [ ]:
from ai_evolution.papers_lab import positional_encoding

def coseno(a, b):
    na = sum(x * x for x in a) ** 0.5
    nb = sum(x * x for x in b) ** 0.5
    return sum(x * y for x, y in zip(a, b)) / (na * nb)

pe = [positional_encoding(p, 16) for p in range(8)]
for p in range(1, 8):
    print(f'cos(PE[0], PE[{p}]) = {coseno(pe[0], pe[p]):+.4f}')

## 14. Desafío autónomo

Implementa las proyecciones aprendidas W_Q, W_K, W_V (aquí ausentes) y entrena el bloque en una tarea de copia. Mide qué aporta cada cabeza haciendo una ablación: desactiva una cabeza y reporta la caída de exactitud.


## 15. Evidencia de aprendizaje

Guarda: entropía con y sin escala, la matriz causal triangular, la tabla de complejidad y una frase sobre qué compró y qué pagó el Transformer.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P08_transformer/README.md) · evaluación formal: [`assessments/papers/P08_transformer.md`](../../assessments/papers/P08_transformer.md)


## 16. Cierre

El bloque está completo. A partir de aquí, la historia se bifurca: usar solo el encoder (P09) o solo el decoder (P10). Las ocho miniaturas `T01`–`T08` desmontan este bloque pieza por pieza.


## 17. Conexión con el siguiente hito

- P09
- P10

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
